# Cleaning and building the model tables

This is where the raw files become the two tables I actually use: the monthly panel for the forecasting work and the pooled Findex records for the adoption model. The jobs are to work out agent density and monthly account ownership, build the lag features, and recode the survey answers to 0/1.

## Setup (local or Google Colab)

On Colab this pulls the project data and code from GitHub. Locally it uses the repo folder you already have.

In [1]:
# Works whether you run this locally or on Google Colab.
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.isdir('data'):
    import subprocess
    subprocess.run(['git','clone','-q','https://github.com/bcudjoe/mobile-money-ghana-forecasting.git'])
    os.chdir('mobile-money-ghana-forecasting')
print('Colab' if IN_COLAB else 'local')

local


In [2]:
import pandas as pd, numpy as np, os
RAW='data/raw/'; OUT='data/processed/'; os.makedirs(OUT, exist_ok=True)

## Load the monthly panel

Read the monthly file and put it in date order. The derived columns (lags, agent density, account ownership) are built further down.

In [3]:
m = pd.read_excel(RAW+'monthly_series.xlsx')
m['date']=pd.to_datetime(m['date']); m=m.sort_values('date').reset_index(drop=True)
print(f'{len(m)} months from {m.date.min():%Y-%m} to {m.date.max():%Y-%m}; {m.shape[1]} columns')

84 months from 2019-01 to 2025-12; 21 columns


## Work out agent density

Agents per 100,000 adults. The population figures are annual, so I interpolate them to monthly first and back-fill the ends so no month is left empty.

In [4]:
pop=pd.read_excel(RAW+'agent_density_population.xlsx')
pop['date']=pd.to_datetime(pop['year'].astype(str)+'-07-01')
cal=pd.date_range(m.date.min(),m.date.max(),freq='MS')
adult=(pop.set_index('date')['adult_population']
         .reindex(pop.set_index('date').index.union(cal)).interpolate('time').reindex(cal).ffill().bfill())
m['agent_density']=m['mm_agents_active'].values/(adult.values/100000)
print('agent_density %.1f -> %.1f'%(m.agent_density.iloc[0], m.agent_density.iloc[-1]))

agent_density 1037.1 -> 2304.0


## Fill in monthly account ownership

Only a handful of survey-year points exist (2011 to 2024), so I interpolate across them. This is a slow-moving structural number, not a monthly signal.

In [5]:
own=pd.read_excel(RAW+'yearly_account_ownership.xlsx')
own['date']=pd.to_datetime(own['YEAR'].astype(str)+'-07-01')
s=own.set_index('date')['account_ownership'].sort_index()
m['account_ownership']=(s.reindex(s.index.union(cal)).interpolate('time').reindex(cal).ffill().bfill()).values
print('account_ownership %.1f - %.1f'%(m.account_ownership.min(), m.account_ownership.max()))

account_ownership 61.7 - 81.2


## Build the time features

Lags, rolling means, year-on-year growth, month, and a running time index, plus a check on the e-levy flag.

In [6]:
for L in range(1,13):
    m[f'mm_value_lag{L}']=m['mm_value'].shift(L)
    m[f'mm_active_accts_lag{L}']=m['mm_active_accts'].shift(L)
m['mm_value_roll3']=m['mm_value'].rolling(3).mean()
m['mm_value_roll6']=m['mm_value'].rolling(6).mean()
m['mm_value_yoy']=m['mm_value'].pct_change(12)*100
m['month']=m['date'].dt.month; m['t_index']=np.arange(1,len(m)+1)
print('E-Levy months:', int(m['elevy'].sum()))
m.to_csv(OUT+'monthly_series_clean.csv', index=False); print('saved', m.shape)

E-Levy months: 44
saved (84, 47)


## Pool and recode the two Findex waves

The raw answers use 1 for yes and 2 for no, with 3 and 4 for don't-know and refused. I map yes to 1 and no to 0, send the don't-know and refused answers to missing, then fill the few small gaps.

In [7]:
f21=pd.read_excel(RAW+'findex_ghana_2021.xlsx'); f25=pd.read_excel(RAW+'findex_ghana_2025.xlsx')
fx=pd.concat([f21,f25], ignore_index=True); print('pooled:', len(fx))
# yes(1)->1, no(2)->0; don't-know/refused (3,4) become NaN
for c in ['female','urban','employed','owns_mobile','borrowed_formal','saved_formal']:
    fx[c]=fx[c].map({1:1,2:0})
fx['has_internet']=fx['has_internet'].map({0:0,1:1})
fx.loc[~fx['education'].isin([1,2,3]),'education']=np.nan
fx['age']=fx['age'].fillna(fx['age'].median())
fx['education']=fx['education'].fillna(fx['education'].mode()[0])
for c in ['female','urban','employed','owns_mobile','borrowed_formal','saved_formal','has_internet']:
    fx[c]=fx[c].fillna(fx[c].mode()[0]).astype(int)
assert fx.isna().sum().sum()==0
fx.to_csv(OUT+'findex_ghana_clean.csv', index=False); print('saved', fx.shape)

pooled: 2000
saved (2000, 15)


## Quick checks

Adoption by wave and by income group, just to see the recoding looks sensible.

In [8]:
print((fx.groupby('wave')['adopts_mm'].mean()*100).round(1))
print((fx.groupby('income_quintile')['adopts_mm'].mean()*100).round(1))

wave
2021    67.8
2025    81.8
Name: adopts_mm, dtype: float64
income_quintile
1    56.3
2    66.5
3    73.1
4    79.6
5    88.1
Name: adopts_mm, dtype: float64
